# Scalable Market Basket Analysis Using MapReduce and FP-Growth
**Bidang:** Big Data & Artificial Intelligence  
**Teknologi:** PySpark, MapReduce, FP-Growth, Association Rules

## 1. Arsitektur Sistem
Sistem ini dirancang untuk memproses data transaksi besar menggunakan paradigma komputasi terdistribusi:
1. **Data Ingestion:** Membaca dataset Instacart (order_id, product_name).
2. **MapReduce Layer:** Mentransformasi data mentah menjadi format transaksi (itemsets).
3. **Mining Layer:** Menggunakan algoritma FP-Growth untuk menemukan pola frekuensi.
4. **Knowledge Discovery:** Ekstraksi Association Rules (Support, Confidence, Lift).
5. **Application Layer:** Sistem rekomendasi produk.

## 2. Flowchart Proses
```mermaid
graph TD
A[CSV Dataset] --> B[Mapper: Mapping product ke order_id]
B --> C[Shuffle: Grouping by order_id]
C --> D[Reducer: Aggregating into transaction list]
D --> E[FP-Growth Model Training]
E --> F[Frequent Itemsets Extraction]
F --> G[Association Rules Generation]
G --> H[Recommendation Engine]
```

In [ ]:
import findspark
findspark.init()

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml.fpm import FPGrowth
import pandas as pd
import matplotlib.pyplot as plt
import time

# Inisialisasi Spark Session menggunakan semua core lokal
spark = SparkSession.builder \
    .appName("ScalableMarketBasketAnalysis") \
    .master("local[*]") \
    .config("spark.executor.memory", "4g") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

print(f"Spark Session Berhasil Dibuat dengan Master: {spark.sparkContext.master}")

## 3. Penjelasan MapReduce
- **Mapper:** Mengambil input baris demi baris dan menghasilkan pasangan (K, V) di mana K adalah `order_id` dan V adalah `product_name`.
- **Shuffle:** Spark secara otomatis memindahkan data sehingga semua `product_name` dengan `order_id` yang sama berkumpul di node/core yang sama.
- **Reducer:** Menggabungkan (aggregate) daftar produk menjadi satu list transaksi unik per `order_id`.

In [ ]:
# 4. Simulasi & Implementasi MapReduce
# Load Dataset (Asumsi file sudah ada)
# Untuk demo, kita buat data dummy jika file tidak ditemukan
try:
    df_raw = spark.read.csv("instacart_preprocessed.csv", header=True, inferSchema=True)
except:
    data = [(1, "Apple"), (1, "Bread"), (2, "Apple"), (2, "Milk"), (2, "Bread"), (3, "Milk"), (3, "Bread")]
    df_raw = spark.createDataFrame(data, ["order_id", "product_name"])

start_time = time.time()

# Proses MapReduce menggunakan API High-Level Spark (DataFrame)
transactions = df_raw.groupBy("order_id").agg(F.collect_list("product_name").alias("items"))

transactions.show(5, truncate=False)
print(f"Waktu Preprocessing MapReduce: {time.time() - start_time:.2f} detik")

## 5. Implementasi FP-Growth & Association Rules
FP-Growth lebih efisien daripada Apriori karena hanya melakukan dua kali scan dataset dan membangun FP-Tree.

In [ ]:
# Konfigurasi FP-Growth
fp_growth = FPGrowth(itemsCol="items", minSupport=0.01, minConfidence=0.1)
model = fp_growth.fit(transactions)

# Menampilkan Frequent Itemsets
frequent_itemsets = model.freqItemsets
frequent_itemsets.orderBy(F.desc("freq")).show(10)

# Menampilkan Association Rules
association_rules = model.associationRules
association_rules.orderBy(F.desc("lift")).show(10)

## 6. Recommendation System
Fungsi ini akan memprediksi produk apa yang mungkin dibeli pelanggan berdasarkan item yang ada di keranjang mereka saat ini.

In [ ]:
# Memberikan rekomendasi untuk setiap transaksi yang ada
recommendations = model.transform(transactions)
recommendations.select("items", "prediction").show(10, truncate=False)

## 7. Visualisasi Hasil

In [ ]:
# Visualisasi Top 10 Frequent Itemsets
top_items = frequent_itemsets.toPandas().nlargest(10, 'freq')
plt.figure(figsize=(10, 6))
plt.barh(top_items['items'].apply(lambda x: ', '.join(x)), top_items['freq'], color='skyblue')
plt.xlabel('Frekuensi')
plt.title('Top 10 Frequent Itemsets')
plt.gca().invert_yaxis()
plt.show()

## 8. Evaluasi Performa
Sistem dievaluasi berdasarkan waktu eksekusi dan skalabilitas core lokal.

In [ ]:
execution_time = time.time() - start_time
print(f"Total Execution Time: {execution_time:.2f} seconds")
print(f"Total Transactions Processed: {transactions.count()}")